# 3151-M05 — From regression to logistic regression

In this mini-lab you will move from linear regression to logistic regression on a **binary classification** problem.

**Scenario (DGP & decision context)**

We work for a subscription app that wants to predict whether a user will **churn in the next 30 days** (`churned = 1`) based on usage and support data collected this month.

- Features:
  - `minutes_per_day` — average minutes the user spends in the app per day this month (0–300).
  - `support_tickets_last_month` — count of support tickets opened in the past month (0–5).
  - `months_since_signup` — how many months since the user signed up (1–36).

- Decision: we will **offer a retention discount** to users predicted likely to churn.
- Stakeholders: the company (revenue) and users (fair and non-spammy offers).

**Risks**

False negatives (missing a churning user) lose revenue; false positives waste discounts. If some groups systematically get worse predictions, this raises **fairness** concerns.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix

np.random.seed(3151)

plt.rcParams["figure.figsize"] = (6, 4)


## Synthetic churn dataset

We will simulate a small, realistic-looking dataset (~800 rows).

The *true* data-generating process is logistic:

$$
P(\text{churned}=1 \mid x) = \sigma(w^\top x)
$$

You don't need to touch the code below — but read the comments to connect the simulation to the model from the video.


In [ ]:
n = 800

minutes_per_day = np.random.gamma(shape=4.0, scale=20.0, size=n)  # ~80 mins avg
minutes_per_day = np.clip(minutes_per_day, 0, 300)

support_tickets_last_month = np.random.poisson(lam=0.4, size=n)
support_tickets_last_month = np.clip(support_tickets_last_month, 0, 5)

months_since_signup = np.random.randint(1, 37, size=n)

X = np.column_stack([
    minutes_per_day,
    support_tickets_last_month,
    months_since_signup,
])

# True parameters for the logistic DGP
w_true = np.array([-3.0, -0.02, 0.6, -0.03])  # include bias term

# Add bias feature of 1s
X_with_bias = np.column_stack([np.ones(n), X])

z = X_with_bias @ w_true
p_churn = 1 / (1 + np.exp(-z))
y = np.random.binomial(1, p_churn)

df = pd.DataFrame({
    "minutes_per_day": minutes_per_day,
    "support_tickets_last_month": support_tickets_last_month,
    "months_since_signup": months_since_signup,
    "churned": y,
})

print(df.head())
print("\nClass balance (mean of churned):", df["churned"].mean().round(3))


## Task 1 — Train/test split & *linear regression* baseline

In the video you saw why **linear regression is not ideal for classification**, but it's a useful baseline.

1. Split the data into train and test.
2. Fit a `LinearRegression` model on the training set.
3. Use it to produce *score-like* predictions on the test set.
4. Clip them to `[0, 1]` so we can pretend they are probabilities and threshold at 0.5.
5. Compute accuracy on the test set.

🧠 Concept check: notice that nothing in this baseline forces outputs to live in `[0,1]` or represent valid probabilities.


In [ ]:
# TODO: adjust test_size or random_state if you want to experiment
X = df[["minutes_per_day", "support_tickets_last_month", "months_since_signup"]]
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=3151,
    stratify=y,  # TODO: why is stratify a good idea here?
)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)  # TODO: what loss is this minimizing?

# Raw predictions (may be outside [0, 1])
y_score_lin = lin_reg.predict(X_test)

# Treat them as pseudo-probabilities
y_proba_lin = np.clip(y_score_lin, 0.0, 1.0)  # TODO: what is weird about this?
y_pred_lin = (y_proba_lin >= 0.5).astype(int)

acc_lin = accuracy_score(y_test, y_pred_lin)
print(f"Linear regression baseline accuracy: {acc_lin:.3f}")

# Simple check
assert y_proba_lin.shape == y_test.shape

plt.scatter(
    X_test["minutes_per_day"], y_test,
    alpha=0.4, label="True churn (0/1)"
)
plt.scatter(
    X_test["minutes_per_day"], y_proba_lin,
    alpha=0.4, marker="x", label="Linear 'probability'"
)
plt.xlabel("Minutes per day")
plt.ylabel("Churn or predicted value")
plt.legend()
plt.title("Linear regression baseline")
plt.show()


## Task 2 — Logistic regression model & metrics

Now switch to **logistic regression** and compare it to the linear baseline.

1. Fit a `LogisticRegression` model on the same training data.
2. Compute predicted probabilities $\hat{p}(y=1 \mid x)$ on the **test** set.
3. Compute:
   - accuracy (with a 0.5 threshold),
   - logistic loss / cross-entropy (`log_loss`).
4. Compare these to the linear baseline.

Use the TODO comments as a guide.


In [ ]:
log_reg = LogisticRegression(
    penalty="none",   # TODO: later, try "l2"
    solver="lbfgs",
    max_iter=1000,
)

log_reg.fit(X_train, y_train)

# Predicted probabilities for class 1 (churn)
y_proba_log = log_reg.predict_proba(X_test)[:, 1]
y_pred_log = (y_proba_log >= 0.5).astype(int)

acc_log = accuracy_score(y_test, y_pred_log)
loss_log = log_loss(y_test, y_proba_log)

print(f"Logistic regression accuracy:      {acc_log:.3f}")
print(f"Logistic regression log-loss:     {loss_log:.3f}")
print(f"Linear regression accuracy (T1):  {acc_lin:.3f}")

# Checks
assert np.all((y_proba_log >= 0.0) & (y_proba_log <= 1.0))

fig, ax = plt.subplots()
ax.hist(y_proba_log[y_test == 0], bins=15, alpha=0.6, label="True non-churn (0)")
ax.hist(y_proba_log[y_test == 1], bins=15, alpha=0.6, label="True churn (1)")
ax.set_xlabel("Predicted probability of churn")
ax.set_ylabel("Count")
ax.set_title("Logistic regression probability distributions")
ax.legend()
plt.show()


## Task 3 — Decision threshold & cost-sensitive thinking

Suppose:

- False **negative** (we fail to target a churning user) costs us **$200**.
- False **positive** (we target a non-churning user) costs **$20** in unnecessary discounts.

So we care more about **recall for the positive (churn=1) class** than about precision.

1. Pick a **lower** decision threshold (e.g. 0.3 instead of 0.5).
2. Compute confusion matrices and class-wise recall for both thresholds.
3. Briefly explain in words which threshold you would choose *and why*.

Write your explanation in the small markdown cell after the code.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, recall_score

def evaluate_threshold(threshold):
    y_pred = (y_proba_log >= threshold).astype(int)
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    print(f"Threshold = {threshold:.2f}")
    print(f"  Accuracy: {acc:.3f}")
    print(f"  Recall (churn=1): {rec:.3f}")

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["no churn (0)", "churn (1)"],
        normalize=None,
    )
    plt.title(f"Confusion matrix at threshold={threshold:.2f}")
    plt.show()

# Baseline threshold 0.5
evaluate_threshold(0.5)

# TODO: try a lower threshold such as 0.3 and observe what changes
evaluate_threshold(0.3)

# Quick checks
assert 0.0 <= y_proba_log.min() <= y_proba_log.max() <= 1.0


### Short explanation

🧠 In 2–3 sentences, answer:

- Which threshold would you pick for this decision problem?
- How does your choice relate to the **costs** of false positives vs false negatives?
- What, if anything, did accuracy *fail* to tell you here?


## Stretch tasks (optional, if you have time)

Pick **one** of these and try it. Add small code and a short markdown note with what you found.

1. **Regularization**: change `penalty="l2"` and try different values of `C` in `LogisticRegression`. Does the log-loss on the test set improve or get worse?
2. **Imbalance experiment**: artificially drop some positive examples from the training set (e.g. keep only half of the `churned=1` rows). Re-train and see how the probabilities and thresholds behave.
3. **Calibration curve (harder)**: create a small calibration plot by binning the predicted probabilities into 5–10 bins and plotting average predicted probability vs empirical fraction of positives.


## Reflection

In your own words (bullet points are fine):

- Which **assumption** of logistic regression felt most important in this lab?
- What did the **logistic loss** reveal that accuracy hid?
- If you had 10× more data or compute, what **experiment** would you run next on this problem?
